# 01 — Data Exploration

Explore raw videos, extracted frames, and trajectory data.  
Run cells top to bottom after completing data ingestion.

**Prerequisites**:  
```bash
python src/ingestion/download_ttnet.py --synthetic
python src/processing/extract_frames.py --input data/raw/ --output data/processed/
python src/processing/tracker.py --video data/raw/synthetic/synthetic_samples/synthetic_topspin_000.mp4 --output data/processed/trajectories/
```

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import cv2
from IPython.display import display, Image

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✓')

## 1. Raw Video Inventory

In [ ]:
raw_dir = Path('../data/raw')
videos = list(raw_dir.rglob('*.mp4'))
print(f'Found {len(videos)} videos')

video_info = []
for v in videos:
    cap = cv2.VideoCapture(str(v))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    w = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    h = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release()
    video_info.append({
        'name': v.name[:50],
        'parent': v.parent.name,
        'fps': round(fps, 1),
        'total_frames': int(frames),
        'duration_sec': round(frames / max(fps, 1), 1),
        'resolution': f'{int(w)}x{int(h)}',
        'size_mb': round(v.stat().st_size / 1e6, 1)
    })

df_videos = pd.DataFrame(video_info)
display(df_videos)
print(f'\nTotal duration: {df_videos.duration_sec.sum():.0f}s ({df_videos.duration_sec.sum()/60:.1f} min)')
print(f'Total size: {df_videos.size_mb.sum():.0f} MB')

## 2. Sample Frame Visualization

In [ ]:
if len(videos) > 0:
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax, video in zip(axes.flat, videos[:8]):
        cap = cv2.VideoCapture(str(video))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, total // 3)
        ret, frame = cap.read()
        cap.release()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_small = cv2.resize(frame_rgb, (320, 240))
            ax.imshow(frame_small)
            ax.set_title(video.name[:30], fontsize=8)
        ax.axis('off')
    plt.suptitle('Sample Frames from Raw Videos', fontsize=14)
    plt.tight_layout()
    plt.savefig('../reports/sample_frames.png', bbox_inches='tight')
    plt.show()

## 3. Trajectory Exploration

In [ ]:
traj_dir = Path('../data/processed/trajectories')
traj_files = list(traj_dir.rglob('*_trajectories.json'))
print(f'Found {len(traj_files)} trajectory files')

all_trajectories = []
for jf in traj_files:
    with open(jf) as f:
        data = json.load(f)
    for t in data.get('trajectories', []):
        t['video'] = Path(data['video']).name
        t['fps'] = data.get('fps', 30.0)
        all_trajectories.append(t)

print(f'Total trajectories: {len(all_trajectories)}')
if all_trajectories:
    spin_counts = pd.Series([t.get('spin_label', 'unknown') for t in all_trajectories]).value_counts()
    print('\nSpin label distribution:')
    print(spin_counts.to_string())

In [ ]:
# Visualize trajectories by spin type
SPIN_COLORS = {'topspin': '#2196F3', 'backspin': '#4CAF50', 'sidespin': '#FF9800', 'float': '#9C27B0', 'unknown': '#aaaaaa'}

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
spins = ['topspin', 'backspin', 'sidespin', 'float']

for ax, spin in zip(axes, spins):
    spin_trajs = [t for t in all_trajectories if t.get('spin_label') == spin]
    if not spin_trajs:
        ax.set_title(f'{spin.title()}\n(no data)')
        continue
    
    for t in spin_trajs[:30]:
        pos = np.array(t['positions'])
        ax.plot(pos[:, 0], 1 - pos[:, 1], alpha=0.3, color=SPIN_COLORS[spin], linewidth=1)
    
    # Mean trajectory
    max_len = max(len(np.array(t['positions'])) for t in spin_trajs[:30])
    all_x, all_y = [], []
    for t in spin_trajs[:30]:
        pos = np.array(t['positions'])
        x_r = np.interp(np.linspace(0, 1, 20), np.linspace(0, 1, len(pos)), pos[:, 0])
        y_r = np.interp(np.linspace(0, 1, 20), np.linspace(0, 1, len(pos)), pos[:, 1])
        all_x.append(x_r)
        all_y.append(y_r)
    ax.plot(np.mean(all_x, axis=0), 1 - np.mean(all_y, axis=0),
            color=SPIN_COLORS[spin], linewidth=3, label='mean')
    
    ax.set_title(f'{spin.title()}\n(n={len(spin_trajs)})')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('X (normalized)')

axes[0].set_ylabel('Y (normalized, flipped)')
plt.suptitle('Ball Trajectories by Spin Type', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../reports/trajectories_by_spin.png', bbox_inches='tight')
plt.show()

## 4. Frame Extraction Statistics

In [ ]:
frame_index = Path('../data/processed/frame_index.csv')
if frame_index.exists():
    df_frames = pd.read_csv(frame_index)
    print(f'Total frames extracted: {len(df_frames):,}')
    print(f'Unique videos: {df_frames.video_id.nunique()}')
    display(df_frames.head(5))
    
    # Sharpness distribution
    if 'sharpness' in df_frames.columns:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(df_frames['sharpness'].dropna(), bins=50, color='steelblue', edgecolor='white')
        ax.axvline(50, color='red', linestyle='--', label='Quality threshold (50)')
        ax.set_xlabel('Sharpness (Laplacian variance)')
        ax.set_ylabel('Frame count')
        ax.set_title('Frame Sharpness Distribution')
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print('No frame index found. Run extract_frames.py first.')